# WSO Conversion Propensity Model

Predicts which free tier members are likely to upgrade to paid Academy, identifies the
behavioral drivers of conversion, and produces propensity scores for targeted outreach.

**Method:**
- Logistic regression as the primary model (interpretable coefficients explain the drivers)
- Random Forest as a benchmark to validate the model choice
- Stratified 80/20 train/test split; stratified 5 fold cross validation on the training set
- Evaluated on ROC AUC, precision/recall, and precision@k (appropriate for class imbalance)

**Data required:** `conversion_dataset.csv` (the clean 25 column user level table)


## 1. Upload the dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

import io, pandas as pd
fname = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[fname]))
print(f"Loaded '{fname}'  ->  {df.shape[0]:,} rows x {df.shape[1]} columns")


## 2. Validate the file

In [ ]:
expected = {
    'user_id','profile_source','has_profile','university_name','school_tier','class_year',
    'major_name','major_cat','mentor_booked','live_resume_used','resume_tool_used',
    'course_previews','target_co_searches','logins','content_views','forum_posts',
    'total_events','intent_events','intent_ratio','tenure_days','days_since_active',
    'email_open_rate','email_click_rate','has_email_record','upgraded'
}
missing = expected - set(df.columns)
assert not missing, f"Wrong file? Missing columns: {missing}"
print("Column check passed.")
print(f"Overall conversion rate: {df['upgraded'].mean()*100:.1f}%  ({df['upgraded'].sum()} upgraders)")


## 3. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, classification_report,
                             confusion_matrix)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports ready.")


## 4. Define features and target

`university_name` and `major_name` are display fields kept for the dashboard.
They are too high cardinality to model directly with ~237 positives, so we use
the lower cardinality `school_tier` and `major_cat` columns instead.

Categorical features are one hot encoded with `unknown` as its own level.
Numeric features are standardized so logistic regression coefficients are
comparable in magnitude.


In [ ]:
target = 'upgraded'
drop_cols = ['user_id', 'university_name', 'major_name', target]

categorical = ['profile_source', 'school_tier', 'class_year', 'major_cat']
numeric = [c for c in df.columns if c not in drop_cols + categorical]

X = df[categorical + numeric].copy()
y = df[target].copy()

print("Categorical features:", categorical)
print("Numeric features:    ", numeric)
print(f"\nTotal features going in: {len(categorical) + len(numeric)}")


## 5. Stratified train/test split

80/20 split, stratified on the label so both sets maintain the ~14% conversion rate.
The test set is locked away and only touched once at the end for final metrics.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

print(f"Train: {len(X_train):,} rows, {y_train.sum()} positives ({y_train.mean()*100:.1f}%)")
print(f"Test:  {len(X_test):,} rows, {y_test.sum()} positives ({y_test.mean()*100:.1f}%)")


## 6. Preprocessing pipeline

In [ ]:
preprocess = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop=None), categorical),
])
print("Preprocessor defined: StandardScaler on numeric, OneHotEncoder on categorical.")


## 7. Logistic Regression (with cross validation)

`class_weight='balanced'` upweights the minority class so the model does not simply
predict "nobody converts." L2 regularization keeps coefficients stable at this sample size.


In [ ]:
logreg = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000,
                               random_state=RANDOM_STATE)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_auc = cross_val_score(logreg, X_train, y_train, cv=cv, scoring='roc_auc')

print("Logistic Regression: 5 fold CV ROC AUC on training set:")
print("  fold scores:", np.round(cv_auc, 3))
print(f"  mean AUC:    {cv_auc.mean():.3f}  (+/- {cv_auc.std():.3f})")

logreg.fit(X_train, y_train)


## 8. Random Forest benchmark

If the forest only marginally beats logistic regression on AUC, that validates
shipping the interpretable model. If it wins by a large margin, that signals
nonlinearities worth investigating. Either outcome is a useful finding.


In [ ]:
rf = Pipeline([
    ('prep', preprocess),
    ('clf', RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                   max_depth=None, min_samples_leaf=5,
                                   random_state=RANDOM_STATE, n_jobs=-1)),
])

rf_cv_auc = cross_val_score(rf, X_train, y_train, cv=cv, scoring='roc_auc')
print("Random Forest: 5 fold CV ROC AUC on training set:")
print("  fold scores:", np.round(rf_cv_auc, 3))
print(f"  mean AUC:    {rf_cv_auc.mean():.3f}  (+/- {rf_cv_auc.std():.3f})")

rf.fit(X_train, y_train)

print(f"\nCV comparison:  LogReg {cv_auc.mean():.3f}   vs   RandomForest {rf_cv_auc.mean():.3f}")


## 9. Final evaluation on the held out test set

Both models are evaluated on the test set exactly once. ROC AUC measures overall
ranking quality. Average Precision (area under the precision recall curve) is the
imbalance appropriate companion metric.


In [ ]:
def evaluate(model, name):
    proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    ap  = average_precision_score(y_test, proba)
    print(f"{name}:  ROC AUC = {auc:.3f}   Avg Precision = {ap:.3f}")
    return proba, auc, ap

print("HELD OUT TEST SET PERFORMANCE")
print("=" * 50)
lr_proba, lr_auc, lr_ap = evaluate(logreg, "Logistic Regression")
rf_proba, rf_auc, rf_ap = evaluate(rf, "Random Forest      ")

print("\nLogistic Regression classification report (threshold 0.5):")
print(classification_report(y_test, (lr_proba >= 0.5).astype(int),
                            target_names=['no upgrade','upgrade']))


## 10. Precision@k

In practice, outreach targets a fixed number of top scored users. Precision@k
measures what fraction of the top k users actually convert. This directly
determines whether targeted outreach beats uniform outreach.


In [ ]:
def precision_at_k(y_true, scores, k):
    idx = np.argsort(scores)[::-1][:k]
    return y_true.iloc[idx].mean()

base_rate = y_test.mean()
print(f"Baseline (random) conversion in test set: {base_rate*100:.1f}%\n")
print(f"{'k (top users)':<15}{'LogReg prec@k':<18}{'lift vs random':<15}")
for k in [25, 50, 100, 150]:
    if k <= len(y_test):
        p = precision_at_k(y_test, lr_proba, k)
        print(f"{k:<15}{p*100:>6.1f}%          {p/base_rate:>5.1f}x")


## 11. Conversion drivers: logistic regression coefficients

Because numeric features were standardized, coefficient magnitude reflects
strength of effect and sign reflects direction. Odds ratios (exp of coefficient)
give an intuitive read: an odds ratio of 1.8 means that feature multiplies
the odds of upgrading by 1.8, holding other features constant.


In [ ]:
ohe = logreg.named_steps['prep'].named_transformers_['cat']
cat_names = list(ohe.get_feature_names_out(categorical))
feat_names = numeric + cat_names

coefs = logreg.named_steps['clf'].coef_[0]
coef_df = pd.DataFrame({
    'feature': feat_names,
    'coefficient': coefs,
    'odds_ratio': np.exp(coefs),
}).sort_values('coefficient', ascending=False).reset_index(drop=True)

pd.set_option('display.float_format', lambda v: f"{v:.3f}")
print("TOP 12 drivers pushing conversion UP:")
print(coef_df.head(12).to_string(index=False))
print("\nBOTTOM 8 (pushing conversion DOWN / near zero):")
print(coef_df.tail(8).to_string(index=False))


## 12. Visualize the top drivers

In [ ]:
top = pd.concat([coef_df.head(10), coef_df.tail(5)])
plt.figure(figsize=(9, 7))
colors = ['#1B3A5C' if c > 0 else '#e76f51' for c in top['coefficient']]
plt.barh(top['feature'][::-1], top['coefficient'][::-1], color=colors[::-1])
plt.axvline(0, color='#333', lw=0.8)
plt.title('Conversion Drivers (Standardized Logistic Regression Coefficients)')
plt.xlabel('Coefficient (positive = raises upgrade odds)')
plt.tight_layout()
plt.show()


## 13. Random Forest feature importances

In [ ]:
rf_clf = rf.named_steps['clf']
rf_imp = pd.DataFrame({
    'feature': feat_names,
    'importance': rf_clf.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)

print("Random Forest: top 12 features by importance:")
print(rf_imp.head(12).to_string(index=False))


## 14. Save outputs

Two files are saved for downstream use:
- `propensity_scores.csv`: every user's predicted upgrade probability, used by the
  A/B test to select the targeted outreach group and by the dashboard to visualize
  score distribution.
- `model_coefficients.csv`: the driver table for the dashboard.


In [ ]:
all_proba = logreg.predict_proba(X)[:, 1]
scores_out = df[['user_id','upgraded']].copy()
scores_out['propensity'] = np.round(all_proba, 4)
scores_out = scores_out.sort_values('propensity', ascending=False)
scores_out.to_csv('propensity_scores.csv', index=False)

coef_df.to_csv('model_coefficients.csv', index=False)

print("Wrote propensity_scores.csv and model_coefficients.csv")

from google.colab import files as _f
_f.download('propensity_scores.csv')
_f.download('model_coefficients.csv')
